# 12.10.转置卷积

到目前为止，我们所见到的卷积神经网络层，例如卷积层（ [<span style="color: #ffbf00; font-weight: bold;"> 6.2节 </span>](../06_pypto_convolutional_networks/06.02_conv_layer.ipynb)）和汇聚层（ [<span style="color: #ffbf00; font-weight: bold;"> 6.5节 </span>](../06_pypto_convolutional_networks/06.05_pooling.ipynb)），通常会减少下采样输入图像的空间维度（高和宽）。然而如果输入和输出图像的空间维度相同，在以像素级分类的语义分割中将会很方便。例如，输出像素所处的通道维可以保有输入像素在同一位置上的分类结果。

为了实现这一点，尤其是在空间维度被卷积神经网络层缩小后，我们可以使用另一种类型的卷积神经网络层，它可以增加上采样中间层特征图的空间维度。本节将介绍*转置卷积*（transposed convolution） [[Dumoulin et al., 2016]](https://zh.d2l.ai/chapter_references/zreferences.html#id37)，用于逆转下采样导致的空间尺寸减小。

与 [<span style="color: #ffbf00; font-weight: bold;"> 6.2节 </span>](../06_pypto_convolutional_networks/06.02_conv_layer.ipynb)一样，本节使用 PyPTO 多层循环算子来模拟转置卷积的行为，所有计算均在 NPU 上完成，重点在于理解其前向计算过程。

---

## 环境准备

In [1]:
%matplotlib inline
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"

import pypto
import torch
from torch import nn

---

## 12.10.1.基本操作

让我们暂时忽略通道，从基本的转置卷积开始，设步幅为1且没有填充。假设我们有一个$n_h \times n_w$的输入张量和一个$k_h \times k_w$的卷积核。以步幅为1滑动卷积核窗口，每行$n_w$次，每列$n_h$次，共产生$n_h n_w$个中间结果。每个中间结果都是一个$(n_h + k_h - 1) \times (n_w + k_w - 1)$的张量，初始化为0。为了计算每个中间张量，输入张量中的每个元素都要乘以卷积核，从而使所得的$k_h \times k_w$张量替换中间张量的一部分。请注意，每个中间张量被替换部分的位置与输入张量中元素的位置相对应。最后，所有中间结果相加以获得最终结果。

例如， <span style="color: #ffbf00; font-weight: bold;"> 图12.10.1 </span>解释了如何为$2\times 2$的输入张量计算卷积核为$2\times 2$的转置卷积。

<div style="border: solid 16px #f1f1f8; text-align: center; background-color: #f6f7f9">
<img src="./images/trans_conv.svg">
<p style="margin: 12px 0 4px 0; font-size: 0.9em; color: #555; text-align: center;">图12.10.1 卷积核为 2 × 2 的转置卷积。阴影部分是中间张量的一部分，也是用于计算的输入和卷积核张量元素。</p>
</div>
<br />

下面我们对输入矩阵`X`和卷积核矩阵`K`实现基本的转置卷积运算`trans_conv`。由于 NPU 上的输出张量不便对重叠区域反复“散射累加”，这里采用与原书实现**完全等价**的互相关形式：先将输入四周填充$k_h-1$、$k_w-1$个零，再与旋转$180°$后的卷积核做互相关。

In [2]:
# 基础互相关：对 input 与 kernel 做步幅为1的有效互相关，结果写入 output
@pypto.frontend.function
def _corr2d_valid(
    input:  pypto.Tensor([], pypto.DT_FP32),
    kernel: pypto.Tensor([], pypto.DT_FP32),
    output: pypto.Tensor([], pypto.DT_FP32),
):
    kH, kW = kernel.shape[0], kernel.shape[1]
    H_out = input.shape[0] - kH + 1
    W_out = input.shape[1] - kW + 1
    pypto.set_vec_tile_shapes(8, 8)
    for i_idx in pypto.loop(H_out, name="LOOP_L0_i", idx_name="i_idx"):
        for j_idx in pypto.loop(W_out, name="LOOP_L1_j", idx_name="j_idx"):
            pypto.set_vec_tile_shapes(8, 8)
            window = input[i_idx:i_idx + kH, j_idx:j_idx + kW]
            mul_result = pypto.mul(window, kernel)
            sum_h = pypto.sum(mul_result, dim=0, keepdim=False)
            scalar = pypto.sum(sum_h, dim=0, keepdim=False)
            result_1x1 = pypto.reshape(scalar, [1, 1])
            pypto.assemble(result_1x1, [i_idx, j_idx], output)


# 二维转置卷积：stride 作用于中间结果，padding 作用于输出（见 12.10.2 节）
@pypto.frontend.jit()
def trans_conv2d_kernel(
    X: pypto.Tensor([], pypto.DT_FP32),
    K: pypto.Tensor([], pypto.DT_FP32),
    Y: pypto.Tensor([], pypto.DT_FP32),
    stride_h: int, stride_w: int,
    pad_h: int, pad_w: int,
):
    kH, kW = K.shape[0], K.shape[1]
    H_in, W_in = X.shape[0], X.shape[1]

    pypto.set_vec_tile_shapes(8, 8)

    # ---- 步骤1：将卷积核旋转180° ----
    idx_h = pypto.arange(kH - 1, -1, -1)
    idx_w = pypto.arange(kW - 1, -1, -1)
    idx_h_2d = pypto.expand_clone(pypto.reshape(idx_h, [kH, 1]), [kH, kW])
    idx_w_2d = pypto.expand_clone(pypto.reshape(idx_w, [1, kW]), [kH, kW])
    K_rot = pypto.gather(pypto.gather(K, 0, idx_h_2d), 1, idx_w_2d)

    # ---- 步骤2：在相邻元素之间插入 stride-1 个零（上采样） ----
    H_up = (H_in - 1) * stride_h + 1
    W_up = (W_in - 1) * stride_w + 1
    x_up = pypto.zeros(H_up, W_up, dtype=pypto.DT_FP32)
    for i_idx in pypto.loop(H_in, name="LOOP_L0_i", idx_name="i_idx"):
        for j_idx in pypto.loop(W_in, name="LOOP_L1_j", idx_name="j_idx"):
            elem = pypto.view(X, [1, 1], [i_idx, j_idx])
            pypto.assemble(elem, [i_idx * stride_h, j_idx * stride_w], x_up)

    # ---- 步骤3：将上采样结果放入零画布，四周补充 kH-1 / kW-1 个零 ----
    x_pad = pypto.zeros(H_up + 2 * (kH - 1), W_up + 2 * (kW - 1), dtype=pypto.DT_FP32)
    pypto.assemble(x_up, [kH - 1, kW - 1], x_pad)

    # ---- 步骤4：与旋转后的卷积核做互相关，跳过输出边缘的 pad 行列 ----
    H_out, W_out = Y.shape[0], Y.shape[1]
    pypto.set_vec_tile_shapes(8, 8)
    for p_idx in pypto.loop(H_out, name="LOOP_L2_p", idx_name="p_idx"):
        for q_idx in pypto.loop(W_out, name="LOOP_L3_q", idx_name="q_idx"):
            pypto.set_vec_tile_shapes(8, 8)
            window = pypto.view(x_pad, [kH, kW], [p_idx + pad_h, q_idx + pad_w])
            mul_result = pypto.mul(window, K_rot)
            sum_h = pypto.sum(mul_result, dim=0, keepdim=False)
            scalar = pypto.sum(sum_h, dim=0, keepdim=False)
            result_1x1 = pypto.reshape(scalar, [1, 1])
            pypto.assemble(result_1x1, [p_idx, q_idx], Y)


def trans_conv(X, K):
    """PyPTO 版基本二维转置卷积（步幅1、无填充）"""
    X_c, K_c = X.contiguous(), K.contiguous()
    H, W = X_c.shape
    kH, kW = K_c.shape
    Y = torch.zeros((H + kH - 1, W + kW - 1), dtype=X_c.dtype, device=X_c.device)
    trans_conv2d_kernel(X_c, K_c, Y, 1, 1, 0, 0)
    return Y

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">def trans_conv(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] + h - 1, X.shape[1] + w - 1))
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Y[i: i + h, j: j + w] += X[i, j] * K
    return Y</pre>
  </div>
</details>

与通过卷积核“减少”输入元素的常规卷积（在 [<span style="color: #ffbf00; font-weight: bold;"> 6.2节 </span>](../06_pypto_convolutional_networks/06.02_conv_layer.ipynb)中）相比，转置卷积通过卷积核“广播”输入元素，从而产生大于输入的输出。我们可以通过 <span style="color: #ffbf00; font-weight: bold;"> 图12.10.1 </span>来构建输入张量`X`和卷积核张量`K`从而验证上述实现输出。此实现是基本的二维转置卷积运算。

In [3]:
X = torch.tensor([[0.0, 1.0], [2.0, 3.0]], device='npu:0')
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]], device='npu:0')
trans_conv(X, K)

tensor([[ 0.,  0.,  1.],
        [ 0.,  4.,  6.],
        [ 4., 12.,  9.]], device='npu:0')

或者，当输入`X`和卷积核`K`都是四维张量时，我们可以使用接口与`nn.ConvTranspose2d`一致的`PyPTOConvTranspose2d`模块获得相同的结果。该模块的权重形状同样为$(C_{in}, C_{out}, k_h, k_w)$；多通道时，转置卷积对输入通道维求和，由宿主侧循环逐通道调用单通道算子完成。

In [4]:
def pypto_conv_transpose2d(X, weight, stride=(1, 1), padding=(0, 0)):
    """多通道二维转置卷积：X (N, C_in, H, W)，weight (C_in, C_out, kH, kW)"""
    X_c, W_c = X.contiguous(), weight.contiguous()
    N, C_in, H, W = X_c.shape
    _, C_out, kH, kW = W_c.shape
    s_h, s_w = stride
    p_h, p_w = padding
    H_out = (H - 1) * s_h - 2 * p_h + kH
    W_out = (W - 1) * s_w - 2 * p_w + kW
    Y = torch.zeros((N, C_out, H_out, W_out), dtype=X_c.dtype, device=X_c.device)
    tmp = torch.empty((H_out, W_out), dtype=X_c.dtype, device=X_c.device)
    for n in range(N):
        for ci in range(C_in):
            for co in range(C_out):
                trans_conv2d_kernel(X_c[n, ci], W_c[ci, co], tmp, s_h, s_w, p_h, p_w)
                Y[n, co] += tmp
    return Y


class PyPTOConvTranspose2d(nn.Module):
    """接口对齐 nn.ConvTranspose2d（仅支持 groups=1、bias=False）"""

    def __init__(self, in_channels=1, out_channels=1, kernel_size=(2, 2),
                 stride=(1, 1), padding=(0, 0), bias=False,
                 device='npu:0', dtype=torch.float32):
        super().__init__()
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        if isinstance(stride, int):
            stride = (stride, stride)
        if isinstance(padding, int):
            padding = (padding, padding)
        self.stride = stride
        self.padding = padding
        self.weight = nn.Parameter(torch.rand(
            in_channels, out_channels, kernel_size[0], kernel_size[1],
            device=device, dtype=dtype))

    def forward(self, x):
        return pypto_conv_transpose2d(x, self.weight, self.stride, self.padding)

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">X, K = X.reshape(1, 1, 2, 2), K.reshape(1, 1, 2, 2)
tconv = nn.ConvTranspose2d(1, 1, kernel_size=2, bias=False)
tconv.weight.data = K
tconv(X)</pre>
  </div>
</details>

In [5]:
X, K = X.reshape(1, 1, 2, 2), K.reshape(1, 1, 2, 2)
tconv = PyPTOConvTranspose2d(1, 1, kernel_size=2, bias=False)
tconv.weight.data = K
tconv(X)

tensor([[[[ 0.,  0.,  1.],
          [ 0.,  4.,  6.],
          [ 4., 12.,  9.]]]], device='npu:0')

---

## 12.10.2.填充、步幅和多通道

与常规卷积不同，在转置卷积中，填充被应用于的输出（常规卷积将填充应用于输入）。例如，当将高和宽两侧的填充数指定为1时，转置卷积的输出中将删除第一和最后的行与列。

In [6]:
tconv = PyPTOConvTranspose2d(1, 1, kernel_size=2, padding=1, bias=False)
tconv.weight.data = K
tconv(X)

tensor([[[[4.]]]], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">tconv = nn.ConvTranspose2d(1, 1, kernel_size=2, padding=1, bias=False)
tconv.weight.data = K
tconv(X)</pre>
  </div>
</details>

在转置卷积中，步幅被指定为中间结果（输出），而不是输入。使用 <span style="color: #ffbf00; font-weight: bold;"> 图12.10.1 </span>中相同输入和卷积核张量，将步幅从1更改为2会增加中间张量的高和权重，因此输出张量在 <span style="color: #ffbf00; font-weight: bold;"> 图12.10.2 </span>中。

<div style="border: solid 16px #f1f1f8; text-align: center; background-color: #f6f7f9">
<img src="./images/trans_conv_stride2.svg">
<p style="margin: 12px 0 4px 0; font-size: 0.9em; color: #555; text-align: center;">图12.10.2 卷积核为2 × 2，步幅为2的转置卷积。阴影部分是中间张量的一部分，也是用于计算的输入和卷积核张量元素。</p>
</div>
<br />

以下代码可以验证 <span style="color: #ffbf00; font-weight: bold;"> 图12.10.2 </span>中步幅为2的转置卷积的输出。在`trans_conv2d_kernel`中，这体现为步骤2在上采样画布的相邻元素之间插入$stride-1$个零。

In [7]:
tconv = PyPTOConvTranspose2d(1, 1, kernel_size=2, stride=2, bias=False)
tconv.weight.data = K
tconv(X)

tensor([[[[0., 0., 0., 1.],
          [0., 0., 2., 3.],
          [0., 2., 0., 3.],
          [4., 6., 6., 9.]]]], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">tconv = nn.ConvTranspose2d(1, 1, kernel_size=2, stride=2, bias=False)
tconv.weight.data = K
tconv(X)</pre>
  </div>
</details>

对于多个输入和输出通道，转置卷积与常规卷积以相同方式运作。假设输入有$c_i$个通道，且转置卷积为每个输入通道分配了一个$k_h\times k_w$的卷积核张量。当指定多个输出通道时，每个输出通道将有一个$c_i\times k_h\times k_w$的卷积核。

同样，如果我们将$\mathsf{X}$代入卷积层$f$来输出$\mathsf{Y}=f(\mathsf{X})$，并创建一个与$f$具有相同的超参数、但输出通道数量是$\mathsf{X}$中通道数的转置卷积层$g$，那么$g(Y)$的形状将与$\mathsf{X}$相同。下面的示例可以解释这一点，其中常规卷积同样由 PyPTO 多层循环算子实现（思路同 [<span style="color: #ffbf00; font-weight: bold;"> 6.3节 </span>](../06_pypto_convolutional_networks/06.03_padding_and_strides.ipynb)）。

In [ ]:
@pypto.frontend.function
def _cross2d(
    input:  pypto.Tensor([], pypto.DT_FP32),
    kernel: pypto.Tensor([], pypto.DT_FP32),
    output: pypto.Tensor([], pypto.DT_FP32),
    stride_h: int, stride_w: int,
):
    kH, kW = kernel.shape[0], kernel.shape[1]
    H_out = (input.shape[0] - kH) // stride_h + 1
    W_out = (input.shape[1] - kW) // stride_w + 1
    pypto.set_vec_tile_shapes(8, 8)
    for i_idx in pypto.loop(H_out, name="LOOP_L0_i", idx_name="i_idx"):
        for j_idx in pypto.loop(W_out, name="LOOP_L1_j", idx_name="j_idx"):
            pypto.set_vec_tile_shapes(8, 8)
            i_start = i_idx * stride_h
            j_start = j_idx * stride_w
            window = pypto.view(input, [kH, kW], [i_start, j_start])
            mul_result = pypto.mul(window, kernel)
            sum_h = pypto.sum(mul_result, dim=0, keepdim=False)
            scalar = pypto.sum(sum_h, dim=0, keepdim=False)
            result_1x1 = pypto.reshape(scalar, [1, 1])
            pypto.assemble(result_1x1, [i_idx, j_idx], output)


@pypto.frontend.jit()
def corr2d_kernel(
    input:  pypto.Tensor([], pypto.DT_FP32),
    kernel: pypto.Tensor([], pypto.DT_FP32),
    output: pypto.Tensor([], pypto.DT_FP32),
    stride_h: int, stride_w: int,
    pad_top: int, pad_bottom: int, pad_left: int, pad_right: int,
):
    # 将输入放入零画布中完成填充
    pypto.set_vec_tile_shapes(8, 8)
    padded_h = input.shape[0] + pad_top + pad_bottom
    padded_w = input.shape[1] + pad_left + pad_right
    padded = pypto.zeros(padded_h, padded_w, dtype=pypto.DT_FP32)
    pypto.assemble(input, [pad_top, pad_left], padded)

    _cross2d(padded, kernel, output, stride_h, stride_w)


def pypto_conv2d(X, weight, stride=(1, 1), padding=(0, 0)):
    """多通道常规卷积：X (N, C_in, H, W)，weight (C_out, C_in, kH, kW)"""
    X_c, W_c = X.contiguous(), weight.contiguous()
    N, C_in, H, W = X_c.shape
    C_out, _, kH, kW = W_c.shape
    s_h, s_w = stride
    p_h, p_w = padding
    H_out = (H + 2 * p_h - kH) // s_h + 1
    W_out = (W + 2 * p_w - kW) // s_w + 1
    Y = torch.zeros((N, C_out, H_out, W_out), dtype=X_c.dtype, device=X_c.device)
    tmp = torch.empty((H_out, W_out), dtype=X_c.dtype, device=X_c.device)
    for n in range(N):
        for co in range(C_out):
            acc = torch.zeros((H_out, W_out), dtype=X_c.dtype, device=X_c.device)
            for ci in range(C_in):
                corr2d_kernel(X_c[n, ci], W_c[co, ci], tmp, s_h, s_w, p_h, p_h, p_w, p_w)
                acc += tmp
            Y[n, co] = acc
    return Y


# 输入有10个通道，经常规卷积映射为20个通道，再经转置卷积映射回10个通道
X = torch.rand(size=(1, 10, 16, 16), device='npu:0')
conv_weight = torch.rand(size=(20, 10, 5, 5), device='npu:0')    # Conv2d 权重布局 (C_out, C_in, kH, kW)
tconv_weight = torch.rand(size=(20, 10, 5, 5), device='npu:0')   # ConvTranspose2d 权重布局 (C_in, C_out, kH, kW)
Y = pypto_conv2d(X, conv_weight, stride=(3, 3), padding=(2, 2))
pypto_conv_transpose2d(Y, tconv_weight, stride=(3, 3), padding=(2, 2)).shape == X.shape

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">X = torch.rand(size=(1, 10, 16, 16))
conv = nn.Conv2d(10, 20, kernel_size=5, padding=2, stride=3)
tconv = nn.ConvTranspose2d(20, 10, kernel_size=5, padding=2, stride=3)
tconv(conv(X)).shape == X.shape</pre>
  </div>
</details>

---

## 12.10.3.与矩阵变换的联系

转置卷积为何以矩阵变换命名呢？让我们首先看看如何使用矩阵乘法来实现卷积。在下面的示例中，我们定义了一个$3\times 3$的输入`X`和$2\times 2$卷积核`K`，然后使用 PyPTO 实现的`corr2d`函数计算卷积输出`Y`。

In [ ]:
@pypto.frontend.jit()
def corr2d_forward_kernel(
    X: pypto.Tensor([], pypto.DT_FP32),
    K: pypto.Tensor([], pypto.DT_FP32),
    Y: pypto.Tensor([], pypto.DT_FP32),
):
    _corr2d_valid(X, K, Y)


def corr2d(X, K):
    """PyPTO 版二维互相关"""
    X_c, K_c = X.contiguous(), K.contiguous()
    H, W = X_c.shape
    kH, kW = K_c.shape
    Y = torch.zeros((H - kH + 1, W - kW + 1), dtype=X_c.dtype, device=X_c.device)
    corr2d_forward_kernel(X_c, K_c, Y)
    return Y


X = torch.arange(9.0, device='npu:0').reshape(3, 3)
K = torch.tensor([[1.0, 2.0], [3.0, 4.0]], device='npu:0')
Y = corr2d(X, K)
Y

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">X = torch.arange(9.0).reshape(3, 3)
K = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
Y = corr2d(X, K)
Y</pre>
  </div>
</details>

接下来，我们将卷积核`K`重写为包含大量0的稀疏权重矩阵`W`。权重矩阵的形状是（$4$，$9$），其中非0元素来自卷积核`K`。

In [ ]:
def kernel2matrix(K):
    k, W = torch.zeros(5, device=K.device), torch.zeros((4, 9), device=K.device)
    k[:2], k[3:5] = K[0, :], K[1, :]
    W[0, :5], W[1, 1:6], W[2, 3:8], W[3, 4:] = k, k, k, k
    return W


W = kernel2matrix(K)
W

逐行连结输入`X`，获得了一个长度为9的矢量。然后，`W`的矩阵乘法和向量化的`X`给出了一个长度为4的向量。重塑它之后，可以获得与上面的原始卷积操作所得相同的结果`Y`：我们刚刚使用矩阵乘法实现了卷积。

In [ ]:
Y == torch.matmul(W, X.reshape(-1)).reshape(2, 2)

同样，我们可以使用矩阵乘法来实现转置卷积。在下面的示例中，我们将上面的常规卷积$2 \times 2$的输出`Y`作为转置卷积的输入。想要通过矩阵相乘来实现它，我们只需要将权重矩阵`W`的形状转置为$(9, 4)$。

In [ ]:
Z = trans_conv(Y, K)
Z == torch.matmul(W.T, Y.reshape(-1)).reshape(3, 3)

抽象来看，给定输入向量$\mathbf{x}$和权重矩阵$\mathbf{W}$，卷积的前向传播函数可以通过将其输入与权重矩阵相乘并输出向量$\mathbf{y}=\mathbf{W}\mathbf{x}$来实现。由于反向传播遵循链式法则和$\nabla_{\mathbf{x}}\mathbf{y}=\mathbf{W}^\top$，卷积的反向传播函数可以通过将其输入与转置的权重矩阵$\mathbf{W}^\top$相乘来实现。因此，转置卷积层能够交换卷积层的正向传播函数和反向传播函数：它的正向传播和反向传播函数将输入向量分别与$\mathbf{W}^\top$和$\mathbf{W}$相乘。


---

## 12.10.4.小结

* 与通过卷积核减少输入元素的常规卷积相反，转置卷积通过卷积核广播输入元素，从而产生形状大于输入的输出。
* 如果我们将$\mathsf{X}$输入卷积层$f$来获得输出$\mathsf{Y}=f(\mathsf{X})$并创造一个与$f$有相同的超参数、但输出通道数是$\mathsf{X}$中通道数的转置卷积层$g$，那么$g(Y)$的形状将与$\mathsf{X}$相同。
* 我们可以使用矩阵乘法来实现卷积。转置卷积层能够交换卷积层的正向传播函数和反向传播函数。
* 在 PyPTO 中，转置卷积可以用多层循环算子模拟：上采样（元素间插零）、零填充后与旋转$180°$的卷积核做互相关，与散射累加的定义完全等价。


---

## 12.10.5.练习

1. 在 [<span style="color: #ffbf00; font-weight: bold;"> 12.10.3节 </span>](./12.10_transposed_conv.ipynb)中，卷积输入`X`和转置的卷积输出`Z`具有相同的形状。他们的数值也相同吗？为什么？
2. 使用矩阵乘法来实现卷积是否有效率？为什么？
3. 尝试修改`trans_conv2d_kernel`的调用参数，观察不同`stride`和`padding`下输出形状的变化，并验证其是否满足$(n_h-1)s_h-k_{h}\cdot 2+p_h\cdot(-2)+k_h$形式的转置卷积输出形状公式。

参考答案详见 [answers/12.10_reference_answer](./answers/12.10_reference_answer.ipynb)。

### 12.10.5.1.参考答案（PyPTO）


In [ ]:
!cat answers/txt/12.10_reference_answer_pypto.txt

### 12.10.5.2.参考答案（PyTorch）


In [ ]:
!cat answers/txt/12.10_reference_answer_pytorch.txt